In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
!pip install -q timm

import os
import torch
import timm
import numpy as np
from PIL import Image

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score

DATASET_PATH = "/content/drive/MyDrive/Balanced_Fetal_Brain_Dataset"
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])


full_dataset = datasets.ImageFolder(
    DATASET_PATH,
    transform=train_transform
)

num_classes = len(full_dataset.classes)

print("Classes:", num_classes)
print("Images :", len(full_dataset))

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size]
)

val_dataset.dataset.transform = val_transform

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = nn.functional.cross_entropy(
            inputs,
            targets,
            reduction='none'
        )

        pt = torch.exp(-ce)

        loss = self.alpha * (1-pt)**self.gamma * ce

        return loss.mean()


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = timm.create_model(
    "swin_tiny_patch4_window7_224",
    pretrained=True,
    num_classes=num_classes
)

model = model.to(device)


criterion = FocalLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)

best_acc = 0

for epoch in range(20):

    model.train()

    train_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    scheduler.step()

    model.eval()

    preds = []
    truths = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)

            outputs = model(images)

            pred = outputs.argmax(1).cpu().numpy()

            preds.extend(pred)
            truths.extend(labels.numpy())

    acc = accuracy_score(truths, preds)

    print(
        f"Epoch {epoch+1} | "
        f"Loss {train_loss:.4f} | "
        f"Val Acc {acc:.4f}"
    )

    if acc > best_acc:

        best_acc = acc

        torch.save(
            model.state_dict(),
            "best_fetal_swin.pth"
        )

        print("Model Saved")

print(f"Best Validation Accuracy: {best_acc*100:.2f}%")

Classes: 15
Images : 1916


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Epoch 1 | Loss 60.0692 | Val Acc 0.7188
Model Saved
Epoch 2 | Loss 10.3477 | Val Acc 0.9297
Model Saved
Epoch 3 | Loss 2.9581 | Val Acc 0.9297
Epoch 4 | Loss 1.8076 | Val Acc 0.9219
Epoch 5 | Loss 1.2236 | Val Acc 0.9531
Model Saved
Epoch 6 | Loss 1.2993 | Val Acc 0.9609
Model Saved
Epoch 7 | Loss 0.9892 | Val Acc 0.9479
Epoch 8 | Loss 0.6052 | Val Acc 0.9661
Model Saved
Epoch 9 | Loss 0.3189 | Val Acc 0.9740
Model Saved
Epoch 10 | Loss 0.3532 | Val Acc 0.9740
Epoch 11 | Loss 0.5372 | Val Acc 0.9740
Epoch 12 | Loss 0.1999 | Val Acc 0.9818
Model Saved
Epoch 13 | Loss 0.1542 | Val Acc 0.9714
Epoch 14 | Loss 0.1003 | Val Acc 0.9740
Epoch 15 | Loss 0.0809 | Val Acc 0.9818
Epoch 16 | Loss 0.1415 | Val Acc 0.9844
Model Saved
Epoch 17 | Loss 0.1097 | Val Acc 0.9870
Model Saved
Epoch 18 | Loss 0.1121 | Val Acc 0.9844
Epoch 19 | Loss 0.1150 | Val Acc 0.9844
Epoch 20 | Loss 0.0903 | Val Acc 0.9844
Best Validation Accuracy: 98.70%


In [12]:
torch.save({
    'model_state_dict': model.state_dict(),
    'classes': full_dataset.classes,
    'accuracy': best_acc
}, 'fetal_brain_swin_best.pth')

In [13]:
from google.colab import files
files.download('fetal_brain_swin_best.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
from sklearn.metrics import classification_report

print(classification_report(
    truths,
    preds,
    target_names=full_dataset.classes
))

                           precision    recall  f1-score   support

anold_chiari_malformation       1.00      1.00      1.00        18
           arachnoid_cyst       0.96      1.00      0.98        26
    cerebellah_hypoplasia       1.00      0.92      0.96        25
           cisterna_magna       1.00      1.00      1.00        23
            colphocephaly       0.90      1.00      0.95        18
            encephalocele       1.00      1.00      1.00        17
        holoprosencephaly       1.00      1.00      1.00        20
            hydracenphaly       1.00      1.00      1.00        18
  intracranial_hemorrhage       1.00      0.95      0.97        19
       intracranial_tumor       1.00      1.00      1.00        23
    mild_ventriculomegaly       1.00      0.98      0.99        42
moderate_ventriculomegaly       0.93      0.95      0.94        43
                   normal       1.00      1.00      1.00        45
             polencephaly       1.00      1.00      1.00     